# Stripe Payout Reconciliation Analysis

Analysis of Stripe payout data (2024-01-01 to 2025-10-17)

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✓ Libraries loaded successfully!")

## 1. Load Data

In [ ]:
# Load the Stripe payout reconciliation data
df = pd.read_csv('../data/raw/Itemized_payout_reconciliation_USD_2024-01-01_to_2025-10-17_America-Chicago.csv', 
                 low_memory=False)

print(f"✓ Loaded {len(df):,} transactions")
print(f"✓ Date range: 2024-01-01 to 2025-10-17")
print(f"✓ Columns: {len(df.columns)}")

df.head()

## 2. Data Cleaning

In [ ]:
# Convert timestamps
df['created_utc'] = pd.to_datetime(df['created_utc'])
df['available_on_utc'] = pd.to_datetime(df['available_on_utc'])

# Extract date components
df['created_date'] = df['created_utc'].dt.date
df['created_year'] = df['created_utc'].dt.year
df['created_month'] = df['created_utc'].dt.month
df['created_month_name'] = df['created_utc'].dt.strftime('%Y-%m')
df['created_day_of_week'] = df['created_utc'].dt.day_name()
df['created_hour'] = df['created_utc'].dt.hour

print("✓ Timestamps converted and date components extracted")
print(f"\nDate range: {df['created_date'].min()} to {df['created_date'].max()}")

## 3. Financial Overview

In [ ]:
# Calculate key metrics
total_gross = df['gross'].sum()
total_fees = df['fee'].sum()
total_net = df['net'].sum()
avg_transaction = df['gross'].mean()
median_transaction = df['gross'].median()
fee_rate = (total_fees / total_gross * 100) if total_gross > 0 else 0

print("="*60)
print("FINANCIAL OVERVIEW")
print("="*60)
print(f"Total Gross Revenue:    ${total_gross:>15,.2f}")
print(f"Total Stripe Fees:      ${total_fees:>15,.2f}")
print(f"Total Net Revenue:      ${total_net:>15,.2f}")
print(f"")
print(f"Average Transaction:    ${avg_transaction:>15,.2f}")
print(f"Median Transaction:     ${median_transaction:>15,.2f}")
print(f"Average Fee Rate:       {fee_rate:>15,.2f}%")
print(f"Total Transactions:     {len(df):>15,}")
print("="*60)

## 4. Revenue by Category

In [ ]:
# Group by reporting category
category_summary = df.groupby('reporting_category').agg({
    'gross': ['sum', 'count', 'mean']
}).round(2)
category_summary.columns = ['Total Gross', 'Count', 'Avg Amount']
category_summary = category_summary.sort_values('Total Gross', ascending=False)

print("\nRevenue by Reporting Category:")
print(category_summary)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart - revenue
category_summary['Total Gross'].plot(kind='barh', ax=ax1, color='steelblue')
ax1.set_title('Total Revenue by Category', fontsize=14, fontweight='bold')
ax1.set_xlabel('Revenue ($)', fontsize=12)
ax1.grid(axis='x', alpha=0.3)

# Bar chart - count
category_summary['Count'].plot(kind='barh', ax=ax2, color='coral')
ax2.set_title('Transaction Count by Category', fontsize=14, fontweight='bold')
ax2.set_xlabel('Count', fontsize=12)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Monthly Revenue Trends

In [ ]:
# Monthly aggregation
monthly = df.groupby('created_month_name').agg({
    'gross': 'sum',
    'fee': 'sum',
    'net': 'sum',
    'balance_transaction_id': 'count'
}).round(2)
monthly.columns = ['Gross Revenue', 'Fees', 'Net Revenue', 'Transactions']

print("\nMonthly Revenue Summary:")
print(monthly)

# Interactive plot with Plotly
fig = go.Figure()

fig.add_trace(go.Bar(
    x=monthly.index,
    y=monthly['Gross Revenue'],
    name='Gross Revenue',
    marker_color='lightblue'
))

fig.add_trace(go.Bar(
    x=monthly.index,
    y=monthly['Fees'],
    name='Fees',
    marker_color='coral'
))

fig.add_trace(go.Scatter(
    x=monthly.index,
    y=monthly['Net Revenue'],
    name='Net Revenue',
    mode='lines+markers',
    line=dict(color='green', width=3),
    marker=dict(size=8)
))

fig.update_layout(
    title='Monthly Revenue Trends',
    xaxis_title='Month',
    yaxis_title='Amount ($)',
    height=500,
    hovermode='x unified'
)

fig.show()

## 6. Top Customers Analysis

In [ ]:
# Top customers by revenue
top_customers = df[df['customer_name'].notna()].groupby('customer_name').agg({
    'gross': 'sum',
    'balance_transaction_id': 'count',
    'fee': 'sum'
}).round(2)
top_customers.columns = ['Total Revenue', 'Transactions', 'Total Fees']
top_customers = top_customers.sort_values('Total Revenue', ascending=False).head(15)

print("\nTop 15 Customers by Revenue:")
print(top_customers)

# Visualize
plt.figure(figsize=(12, 8))
plt.barh(range(len(top_customers)), top_customers['Total Revenue'], color='steelblue')
plt.yticks(range(len(top_customers)), top_customers.index)
plt.xlabel('Total Revenue ($)', fontsize=12)
plt.title('Top 15 Customers by Revenue', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Program Performance

In [ ]:
# Analyze programs from metadata
programs = df[df['payment_metadata[Program]'].notna()].groupby('payment_metadata[Program]').agg({
    'gross': ['sum', 'count', 'mean'],
    'fee': 'sum'
}).round(2)
programs.columns = ['Total Revenue', 'Transactions', 'Avg Transaction', 'Total Fees']
programs = programs.sort_values('Total Revenue', ascending=False).head(10)

print("\nTop 10 Programs by Revenue:")
print(programs)

# Pie chart
fig = px.pie(
    values=programs['Total Revenue'],
    names=programs.index,
    title='Revenue Distribution by Program (Top 10)'
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

## 8. Payment Methods Analysis

In [ ]:
# Payment method breakdown
payment_methods = df.groupby('payment_method_type').agg({
    'gross': ['sum', 'count'],
    'fee': 'sum'
}).round(2)
payment_methods.columns = ['Total Revenue', 'Count', 'Total Fees']
payment_methods = payment_methods.sort_values('Total Revenue', ascending=False)

print("\nPayment Method Summary:")
print(payment_methods)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart - by count
ax1.pie(payment_methods['Count'], labels=payment_methods.index, autopct='%1.1f%%', startangle=90)
ax1.set_title('Transactions by Payment Method', fontsize=14, fontweight='bold')

# Pie chart - by revenue
ax2.pie(payment_methods['Total Revenue'], labels=payment_methods.index, autopct='%1.1f%%', startangle=90)
ax2.set_title('Revenue by Payment Method', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Card Brand Analysis

In [ ]:
# Card brand analysis (for card payments only)
card_data = df[df['card_brand'].notna()]
card_brands = card_data.groupby('card_brand').agg({
    'gross': ['sum', 'count', 'mean'],
    'fee': 'sum'
}).round(2)
card_brands.columns = ['Total Revenue', 'Count', 'Avg Transaction', 'Total Fees']
card_brands = card_brands.sort_values('Total Revenue', ascending=False)

print("\nCard Brand Summary:")
print(card_brands)

# Visualize
fig = px.bar(
    x=card_brands.index,
    y=card_brands['Total Revenue'],
    title='Revenue by Card Brand',
    labels={'x': 'Card Brand', 'y': 'Total Revenue ($)'},
    text=card_brands['Count']
)
fig.update_traces(texttemplate='%{text} txns', textposition='outside')
fig.show()

## 10. Transaction Timing Analysis

In [ ]:
# Day of week analysis
dow_analysis = df.groupby('created_day_of_week').agg({
    'gross': ['sum', 'count', 'mean']
}).round(2)
dow_analysis.columns = ['Total Revenue', 'Count', 'Avg Amount']

# Reorder days
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_analysis = dow_analysis.reindex([d for d in day_order if d in dow_analysis.index])

print("\nTransactions by Day of Week:")
print(dow_analysis)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

dow_analysis['Count'].plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Transactions by Day of Week', fontsize=14, fontweight='bold')
ax1.set_ylabel('Transaction Count', fontsize=12)
ax1.set_xlabel('Day', fontsize=12)
ax1.grid(axis='y', alpha=0.3)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

dow_analysis['Total Revenue'].plot(kind='bar', ax=ax2, color='coral')
ax2.set_title('Revenue by Day of Week', fontsize=14, fontweight='bold')
ax2.set_ylabel('Revenue ($)', fontsize=12)
ax2.set_xlabel('Day', fontsize=12)
ax2.grid(axis='y', alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Hour of day analysis
hour_analysis = df.groupby('created_hour').agg({
    'gross': ['sum', 'count']
}).round(2)
hour_analysis.columns = ['Total Revenue', 'Count']

print("\nTransactions by Hour of Day:")
print(hour_analysis.head(10))

# Visualize
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(hour_analysis.index, hour_analysis['Count'], marker='o', linewidth=2, markersize=8)
ax.set_title('Transactions by Hour of Day (UTC)', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour (0-23)', fontsize=12)
ax.set_ylabel('Transaction Count', fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

## 11. Fee Analysis

In [ ]:
# Calculate fee percentages
df['fee_percentage'] = (df['fee'] / df['gross'] * 100).round(2)

# Fee stats
print("\nFee Statistics:")
print(f"Average Fee Rate: {df['fee_percentage'].mean():.2f}%")
print(f"Median Fee Rate: {df['fee_percentage'].median():.2f}%")
print(f"Min Fee Rate: {df['fee_percentage'].min():.2f}%")
print(f"Max Fee Rate: {df['fee_percentage'].max():.2f}%")

# Visualize fee distribution
plt.figure(figsize=(14, 5))
plt.hist(df['fee_percentage'], bins=50, edgecolor='black', alpha=0.7)
plt.title('Fee Percentage Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Fee Percentage (%)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.axvline(df['fee_percentage'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["fee_percentage"].mean():.2f}%')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Growth Analysis

In [ ]:
# Calculate month-over-month growth
monthly_sorted = monthly.sort_index()
monthly_sorted['Revenue_Growth_%'] = monthly_sorted['Gross Revenue'].pct_change() * 100
monthly_sorted['Transaction_Growth_%'] = monthly_sorted['Transactions'].pct_change() * 100

print("\nMonth-over-Month Growth:")
print(monthly_sorted[['Gross Revenue', 'Revenue_Growth_%', 'Transactions', 'Transaction_Growth_%']].tail(12))

# Visualize growth
fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.bar(monthly_sorted.index, monthly_sorted['Gross Revenue'], color='lightblue', alpha=0.7, label='Revenue')
ax1.set_xlabel('Month', fontsize=12)
ax1.set_ylabel('Revenue ($)', fontsize=12, color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
plt.xticks(rotation=45)

ax2 = ax1.twinx()
ax2.plot(monthly_sorted.index, monthly_sorted['Revenue_Growth_%'], color='red', marker='o', linewidth=2, markersize=6, label='Growth %')
ax2.set_ylabel('Growth Rate (%)', fontsize=12, color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)

plt.title('Monthly Revenue and Growth Rate', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 13. Summary Report

In [ ]:
print("="*70)
print("EXECUTIVE SUMMARY")
print("="*70)
print(f"\n📊 OVERALL METRICS")
print(f"   Total Transactions:     {len(df):>15,}")
print(f"   Total Gross Revenue:    ${total_gross:>15,.2f}")
print(f"   Total Net Revenue:      ${total_net:>15,.2f}")
print(f"   Total Fees:             ${total_fees:>15,.2f} ({fee_rate:.2f}%)")

print(f"\n💳 PAYMENT METHODS")
for method, row in payment_methods.head(3).iterrows():
    pct = (row['Count'] / len(df) * 100)
    print(f"   {method:20s}  {row['Count']:>6,.0f} txns ({pct:5.1f}%)  ${row['Total Revenue']:>12,.2f}")

print(f"\n🏆 TOP 3 CUSTOMERS")
for i, (customer, row) in enumerate(top_customers.head(3).iterrows(), 1):
    print(f"   {i}. {customer:30s} ${row['Total Revenue']:>10,.2f} ({row['Transactions']:.0f} txns)")

print(f"\n📈 TOP 3 PROGRAMS")
for i, (program, row) in enumerate(programs.head(3).iterrows(), 1):
    print(f"   {i}. {program:35s} ${row['Total Revenue']:>10,.2f} ({row['Transactions']:.0f} txns)")

print(f"\n📅 RECENT PERFORMANCE (Last 3 months)")
for month, row in monthly_sorted.tail(3).iterrows():
    print(f"   {month}:  ${row['Gross Revenue']:>12,.2f}  ({row['Transactions']:.0f} txns)")

print(f"\n{'-'*70}")
print(f"Peak Month: {monthly_sorted['Gross Revenue'].idxmax()} - ${monthly_sorted['Gross Revenue'].max():,.2f}")
print(f"Average Monthly Revenue: ${monthly_sorted['Gross Revenue'].mean():,.2f}")
print("="*70)

## Done!

This analysis provides comprehensive insights into your Stripe payout data. You can modify any cell to dive deeper into specific areas.